# Examen : Visualisation de Séries Temporelles avec Bokeh

Cet examen porte sur la visualisation de données temporelles à l'aide de la bibliothèque **Bokeh**.
Nous travaillons sur le jeu de données **"Daily Minimum Temperatures in Melbourne"**.

Dataset : `daily-minimum-temperatures-in-melbourne.csv`

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    DatetimeTickFormatter,
    NumeralTickFormatter,
)
from bokeh.layouts import row, column
from bokeh.transform import factor_cmap

# Activation de l'affichage Bokeh dans le notebook
output_notebook()

In [ ]:
# Chargement du dataset
df = pd.read_csv("../datasets/daily-minimum-temperatures-in-melbourne.csv")

# Renommage des colonnes pour plus de clarté
df.columns = ['Date', 'Temperature']

# Conversion de la colonne Date en format datetime
df['Date'] = pd.to_datetime(df['Date'])

# Nettoyage : suppression des '?' et conversion en numérique
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'])

# Affichage d'un aperçu des données
print(df.head())
print(f"\nDimensions du dataset : {df.shape}")
df

## Question 1 : Graphique en ligne basique des températures

Créer un graphique en ligne montrant l'évolution de la température minimale quotidienne.

- Utiliser 'Date' en abscisse et 'Temperature' en ordonnée.
- Titre : "Daily Minimum Temperatures".
- Axes : "Date" (x) et "Temperature (°C)" (y).
- Ajouter des infobulles affichant la date et la température.
- Activer les outils pan, zoom molette et réinitialisation.

In [ ]:
# Création de la source de données
source_q1 = ColumnDataSource(data={
    'Date': df['Date'],
    'Temperature': df['Temperature']
})

# Création de la figure avec type datetime pour l'axe x
p1 = figure(
    title="Daily Minimum Temperatures",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=950,
    height=400
)

# Tracé de la ligne des températures
ligne_q1 = p1.line(
    x='Date',
    y='Temperature',
    source=source_q1,
    line_width=1.5,
    color='navy',
    alpha=0.7
)

# Configuration de l'infobulle au survol
survol_q1 = HoverTool(
    renderers=[ligne_q1],
    tooltips=[
        ("Date", "@Date{%F}"),
        ("Température", "@Temperature{0.0} °C")
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)

# Ajout de l'infobulle et étiquettes des axes
p1.add_tools(survol_q1)
p1.xaxis.axis_label = "Date"
p1.yaxis.axis_label = "Temperature (°C)"

show(p1)

## Question 2 : Moyenne mobile sur 30 jours

Calculer la moyenne mobile sur 30 jours et la superposer aux données brutes.

- Créer une colonne 'Rolling_Avg' avec la moyenne mobile sur 30 jours.
- Tracer les deux courbes sur le même graphique.
- Utiliser des couleurs et styles de ligne différents.
- Ajouter une légende.
- Ajouter des infobulles avec date, température et moyenne mobile.

In [ ]:
# Calcul de la moyenne mobile sur 30 jours
df['Rolling_Avg'] = df['Temperature'].rolling(window=30, min_periods=1).mean()

# Source de données pour cette question
source_q2 = ColumnDataSource(data={
    'Date': df['Date'],
    'Temperature': df['Temperature'],
    'Rolling_Avg': df['Rolling_Avg']
})

# Figure principale
p2 = figure(
    title="Températures quotidiennes et moyenne mobile (30 jours)",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=950,
    height=380
)

# Ligne des températures brutes (bleu, fine)
ligne_temp = p2.line(
    'Date', 'Temperature',
    source=source_q2,
    line_width=1.2,
    color='steelblue',
    alpha=0.6,
    legend_label='Température'
)

# Ligne de la moyenne mobile (rouge, tirets, plus épaisse)
ligne_avg = p2.line(
    'Date', 'Rolling_Avg',
    source=source_q2,
    line_width=2.5,
    color='crimson',
    line_dash='dashed',
    legend_label='Moyenne mobile 30j'
)

# Infobulle commune aux deux lignes
survol_q2 = HoverTool(
    renderers=[ligne_temp, ligne_avg],
    tooltips=[
        ("Date", "@Date{%F}"),
        ("Température", "@Temperature{0.0} °C"),
        ("Moyenne mobile", "@Rolling_Avg{0.0} °C")
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)

p2.add_tools(survol_q2)
p2.legend.location = "top_left"
p2.xaxis.axis_label = "Date"
p2.yaxis.axis_label = "Temperature (°C)"

show(p2)

## Question 3 : Box plots mensuels

Créer des box plots pour visualiser la distribution des températures par mois.

- Extraire le mois depuis 'Date' et créer la colonne 'Month'.
- Grouper les données par mois.
- Utiliser les éléments box plot de Bokeh.
- Étiqueter l'axe x avec les noms de mois et l'axe y avec "Temperature (°C)".
- Ajouter des infobulles avec min, max, médiane.

In [ ]:
# Extraction du mois (abréviation en 3 lettres)
df['Month'] = df['Date'].dt.month_name().str[:3]
mois_ordre = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Calcul des statistiques par mois
groupes_mois = df.groupby('Month')['Temperature']
stats_mensuelles = groupes_mois.describe().reset_index()
stats_mensuelles['q1'] = groupes_mois.quantile(0.25).values
stats_mensuelles['q2'] = groupes_mois.quantile(0.50).values  # médiane
stats_mensuelles['q3'] = groupes_mois.quantile(0.75).values

# Calcul des moustaches (whiskers)
iqr = stats_mensuelles['q3'] - stats_mensuelles['q1']
stats_mensuelles['upper'] = (stats_mensuelles['q3'] + 1.5 * iqr).clip(upper=stats_mensuelles['max'])
stats_mensuelles['lower'] = (stats_mensuelles['q1'] - 1.5 * iqr).clip(lower=stats_mensuelles['min'])

# Tri par ordre chronologique
stats_mensuelles['Month'] = pd.Categorical(stats_mensuelles['Month'], categories=mois_ordre, ordered=True)
stats_mensuelles = stats_mensuelles.sort_values('Month').reset_index(drop=True)
stats_mensuelles['Month'] = stats_mensuelles['Month'].astype(str)

source_q3 = ColumnDataSource(stats_mensuelles)

# Création de la figure
p3 = figure(
    title="Distribution mensuelle des températures",
    x_range=mois_ordre,
    tools="pan,wheel_zoom,reset",
    width=900,
    height=420
)

# Moustaches supérieures et inférieures
p3.segment(x0='Month', y0='upper', x1='Month', y1='q3', source=source_q3, color='gray')
p3.segment(x0='Month', y0='lower', x1='Month', y1='q1', source=source_q3, color='gray')

# Boîtes : partie supérieure (q2→q3) et partie inférieure (q1→q2)
p3.vbar(x='Month', width=0.6, top='q3', bottom='q2', source=source_q3,
        fill_color='#440154', line_color='white')
p3.vbar(x='Month', width=0.6, top='q2', bottom='q1', source=source_q3,
        fill_color='#21918c', line_color='white')

# Caps (extrémités des moustaches)
p3.rect(x='Month', y='lower', width=0.15, height=0.08, source=source_q3, color='gray')
p3.rect(x='Month', y='upper', width=0.15, height=0.08, source=source_q3, color='gray')

# Infobulle
survol_q3 = HoverTool(tooltips=[
    ("Mois", "@Month"),
    ("Min", "@min{0.0} °C"),
    ("Max", "@max{0.0} °C"),
    ("Médiane", "@q2{0.0} °C")
])

p3.add_tools(survol_q3)
p3.xaxis.axis_label = "Mois"
p3.yaxis.axis_label = "Temperature (°C)"

show(p3)

## Question 4 : Box plots annuels avec coloration

Créer des box plots pour chaque année avec un code couleur basé sur la médiane.

- Extraire l'année depuis 'Date' et créer 'Year'.
- Grouper par année.
- Colorer les boîtes selon la médiane avec factor_cmap.
- Infobulles : année, min, max, médiane, Q1, Q3.
- Outils : pan, zoom molette, réinitialisation.

In [ ]:
# Extraction de l'année
df['Year'] = df['Date'].dt.year.astype(str)

# Statistiques par année
groupes_annee = df.groupby('Year')['Temperature']
stats_annuelles = groupes_annee.describe().reset_index()
stats_annuelles['q1'] = groupes_annee.quantile(0.25).values
stats_annuelles['q2'] = groupes_annee.quantile(0.50).values
stats_annuelles['q3'] = groupes_annee.quantile(0.75).values

# Calcul des whiskers
iqr_y = stats_annuelles['q3'] - stats_annuelles['q1']
stats_annuelles['upper'] = (stats_annuelles['q3'] + 1.5 * iqr_y).clip(upper=stats_annuelles['max'])
stats_annuelles['lower'] = (stats_annuelles['q1'] - 1.5 * iqr_y).clip(lower=stats_annuelles['min'])

# Classification des médianes en 3 niveaux
seuil_bas = stats_annuelles['q2'].quantile(0.33)
seuil_haut = stats_annuelles['q2'].quantile(0.66)
stats_annuelles['median_cat'] = stats_annuelles['q2'].apply(
    lambda v: 'Froid' if v <= seuil_bas else ('Chaud' if v >= seuil_haut else 'Moyen')
)

# Liste ordonnée des années
annees = sorted(stats_annuelles['Year'].tolist())
source_q4 = ColumnDataSource(stats_annuelles)

# Palette de couleurs : bleu (froid) → orange (moyen) → rouge (chaud)
cmap_q4 = factor_cmap('median_cat', palette=['#3b4cc0', '#f7f7f7', '#b40426'],
                       factors=['Froid', 'Moyen', 'Chaud'])

p4 = figure(
    title="Distribution annuelle des températures",
    x_range=annees,
    tools="pan,wheel_zoom,reset",
    width=950,
    height=400
)

# Moustaches
p4.segment(x0='Year', y0='upper', x1='Year', y1='q3', source=source_q4, color='#555555')
p4.segment(x0='Year', y0='lower', x1='Year', y1='q1', source=source_q4, color='#555555')

# Boîte IQR colorée par catégorie de médiane
boite_q4 = p4.vbar(
    x='Year', width=0.55, top='q3', bottom='q1',
    source=source_q4, fill_color=cmap_q4, line_color='black'
)

# Caps
p4.rect(x='Year', y='lower', width=0.1, height=0.1, source=source_q4, color='#555555')
p4.rect(x='Year', y='upper', width=0.1, height=0.1, source=source_q4, color='#555555')

# Infobulle détaillée
survol_q4 = HoverTool(
    renderers=[boite_q4],
    tooltips=[
        ("Année", "@Year"),
        ("Min", "@min{0.0} °C"),
        ("Max", "@max{0.0} °C"),
        ("Médiane", "@q2{0.0} °C"),
        ("Q1", "@q1{0.0} °C"),
        ("Q3", "@q3{0.0} °C"),
        ("Catégorie", "@median_cat")
    ]
)

p4.add_tools(survol_q4)
p4.xaxis.axis_label = "Année"
p4.yaxis.axis_label = "Temperature (°C)"

show(p4)

## Question 5 : Sélection interactive de plage de dates

Créer un graphique interactif avec un curseur de plage de dates.

- Tracer la température en fonction de la date.
- Ajouter un slider de plage de dates (DateRangeSlider).
- Mise à jour dynamique du graphique.
- Infobulles : date et température.
- Outils : pan, zoom molette, réinitialisation.

In [ ]:
from bokeh.models import DateRangeSlider

source_q5 = ColumnDataSource(data={
    'Date': df['Date'],
    'Temperature': df['Temperature']
})

p5 = figure(
    title="Températures minimales interactives",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=950,
    height=400
)

# Tracé de la courbe
p5.line('Date', 'Temperature', source=source_q5, line_width=1.5, color='teal')

# Infobulle
survol_q5 = HoverTool(
    tooltips=[
        ("Date", "@Date{%F}"),
        ("Température", "@Temperature{0.0} °C")
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)
p5.add_tools(survol_q5)
p5.xaxis.axis_label = "Date"
p5.yaxis.axis_label = "Temperature (°C)"

# Création du slider de plage de dates
slider_dates = DateRangeSlider(
    title="Sélectionner la plage de dates",
    start=df['Date'].min(),
    end=df['Date'].max(),
    value=(df['Date'].min(), df['Date'].max()),
    width=900
)

# Liaison JavaScript : le slider contrôle l'axe x du graphique
slider_dates.js_link("value", p5.x_range, "start", attr_selector=0)
slider_dates.js_link("value", p5.x_range, "end", attr_selector=1)

# Affichage du slider au-dessus du graphique
show(column(slider_dates, p5))

## Question 6 : Décomposition de la série temporelle

Décomposer la série temporelle pour visualiser tendance et saisonnalité.

- Rééchantillonner en fréquence mensuelle (moyenne).
- Estimer la tendance avec une moyenne mobile sur 12 mois.
- Calculer la composante saisonnière (données - tendance).
- Créer 3 graphiques alignés : données mensuelles, tendance, saisonnalité.
- Infobulles sur chaque graphique.
- Outils : pan, zoom molette, réinitialisation.

In [ ]:
# Rééchantillonnage mensuel
mensuel = df.set_index('Date')['Temperature'].resample('ME').mean().reset_index()

# Tendance : moyenne mobile centrée sur 12 mois
mensuel['Tendance'] = mensuel['Temperature'].rolling(window=12, min_periods=1, center=True).mean()

# Composante saisonnière
mensuel['Saisonnalite'] = mensuel['Temperature'] - mensuel['Tendance']

source_q6 = ColumnDataSource(mensuel)

# --- Graphique 1 : Données mensuelles brutes ---
p6_1 = figure(
    title="Température moyenne mensuelle",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    width=900, height=300
)
p6_1.line('Date', 'Temperature', source=source_q6, line_width=2, color='#1f77b4')
survol_6a = HoverTool(
    tooltips=[("Date", "@Date{%F}"), ("Température", "@Temperature{0.1f} °C")],
    formatters={"@Date": "datetime"}, mode="vline"
)
p6_1.add_tools(survol_6a)
p6_1.xaxis.axis_label = "Date"
p6_1.yaxis.axis_label = "Température (°C)"

# --- Graphique 2 : Tendance ---
p6_2 = figure(
    title="Composante tendancielle",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    x_range=p6_1.x_range,  # Axe x partagé
    width=900, height=250
)
p6_2.line('Date', 'Tendance', source=source_q6, line_width=2.5, color='#ff7f0e')
survol_6b = HoverTool(
    tooltips=[("Date", "@Date{%F}"), ("Tendance", "@Tendance{0.1f} °C")],
    formatters={"@Date": "datetime"}, mode="vline"
)
p6_2.add_tools(survol_6b)
p6_2.xaxis.axis_label = "Date"
p6_2.yaxis.axis_label = "Tendance"

# --- Graphique 3 : Saisonnalité ---
p6_3 = figure(
    title="Composante saisonnière",
    x_axis_type="datetime",
    tools="pan,wheel_zoom,reset",
    x_range=p6_1.x_range,  # Axe x partagé
    width=900, height=250
)
p6_3.line('Date', 'Saisonnalite', source=source_q6, line_width=2, color='#2ca02c')
survol_6c = HoverTool(
    tooltips=[("Date", "@Date{%F}"), ("Saisonnalité", "@Saisonnalite{0.1f} °C")],
    formatters={"@Date": "datetime"}, mode="vline"
)
p6_3.add_tools(survol_6c)
p6_3.xaxis.axis_label = "Date"
p6_3.yaxis.axis_label = "Saisonnalité"

# Affichage des 3 graphiques empilés
show(column(p6_1, p6_2, p6_3))